## Gold Layer Design and Experimentation
This file is used as an experimentation notebook to design the gold layer - two dimension tables and a fact table - based on the silver notebooks from lab 4. They will then be moved to Lakeflow Declarative Pipeline created in Lab 5.

In [0]:
# Configuration
from pyspark.sql import functions as F, Window
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
CATALOG       = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

In [0]:
spark.sql(f"DROP TABLE {CATALOG}.{GOLD_SCHEMA}.dim_datacenter")
spark.sql(f"DROP TABLE {CATALOG}.{GOLD_SCHEMA}.consumption_hourly")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD_SCHEMA}.dim_date")

In [0]:
# Checking the contents of sensor_data
display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data LIMIT 40"))

In [0]:
# Creating gold schema for the dim_datacenter table

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.dim_datacenter
          (
            dc_key BIGINT GENERATED ALWAYS AS IDENTITY,
            site_id STRING NOT NULL,
            site_name STRING,
            country STRING,
            bidding_zone STRING,
            valid_from TIMESTAMP NOT NULL,
            valid_to TIMESTAMP,
            is_current BOOLEAN NOT NULL,
            CONSTRAINT pk_dim_datacenter PRIMARY KEY (dc_key))
            USING DELTA;
            """)

In [0]:
spark.sql(f"""INSERT OVERWRITE TABLE {CATALOG}.{GOLD_SCHEMA}.dim_datacenter 
          (site_id, site_name, country, bidding_zone, valid_from, valid_to, is_current) 
          SELECT 
          site_id, 
          site_name, 
          country, 
          bidding_zone, 
          valid_from, 
          valid_to, 
          is_current 
          FROM {CATALOG}.{SILVER_SCHEMA}.dim_datacenter""")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dim_datacenter"))

In [0]:
# Creating a fact table for events - hourly consumption
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.consumption_hourly (
  site_id               STRING     NOT NULL,
  bidding_zone          STRING     NOT NULL,
  date                  DATE       NOT NULL,
  hour                  INTEGER    NOT NULL,
  consumption_kwh       DECIMAL(10,4),
  avg_power_kw          DECIMAL(10,2),
  pue                   DECIMAL(4,3),
  cost_per_hour         DECIMAL(10,2)
)
USING DELTA
""")


In [0]:
spark.sql(f"""
          INSERT OVERWRITE {CATALOG}.{GOLD_SCHEMA}.consumption_hourly 
          (site_id, bidding_zone, date, hour, consumption_kwh, avg_power_kw, pue, cost_per_hour) 
          SELECT 
          s.site_id,
          s.bidding_zone,
          DATE(s.timestamp_utc) as date,
          HOUR(s.timestamp_utc) as hour,
          AVG(s.consumption_kwh),
          AVG(s.avg_power_kw),
          AVG(s.pue),
          AVG((s.consumption_kwh * p.price) / 1000) as cost_per_hour
          FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data AS s
          LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.prices AS p
          ON DATE_TRUNC('hour', s.timestamp_utc) = DATE_TRUNC('hour', p.timestamp_utc) 
             AND s.bidding_zone = p.bidding_zone
          GROUP BY s.bidding_zone, s.site_id, DATE(s.timestamp_utc), HOUR(s.timestamp_utc)""")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly ORDER BY date, hour"))


In [0]:
# Some rows have null cost_per_hour - checking how many
display(spark.sql(f"""
SELECT COUNT(*) AS total,
       COUNT(cost_per_hour) AS with_cost,
       COUNT(*) - COUNT(cost_per_hour) AS null_cost
FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly
"""))

In [0]:
spark.sql(f"DROP TABLE {CATALOG}.{GOLD_SCHEMA}.dim_date")
# Creating a dim table with different date grains
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.dim_date (
  date                  DATE     NOT NULL,
  month                 INTEGER     NOT NULL,
  day                   INTEGER     NOT NULL,
  week                  INTEGER     NOT NULL,
  year                  INTEGER     NOT NULL,
  day_of_week           INTEGER  NOT NULL,
  day_of_week_name      STRING   NOT NULL,
  week_of_year          INTEGER  NOT NULL,
  month_name            STRING    NOT NULL,
  quarter               INTEGER  NOT NULL,
  is_weekend            STRING  NOT NULL,
  CONSTRAINT pk_dim_date PRIMARY KEY (date))
USING DELTA
""")


In [0]:
spark.sql(f"""
INSERT OVERWRITE {CATALOG}.{GOLD_SCHEMA}.dim_date
(
    date,
    month,
    day,
    week,
    year,
    day_of_week,
    day_of_week_name,
    week_of_year,
    month_name,
    quarter,
    is_weekend
)
SELECT
    date,
    month(date) AS month,
    day(date) AS day,
    weekofyear(date) AS week,
    year(date) AS year,
    dayofweek(date) AS day_of_week,

    CASE dayofweek(date)
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
        ELSE 'Unknown'
    END AS day_of_week_name,

    weekofyear(date) AS week_of_year,
    date_format(date, 'MMMM') AS month_name,
    quarter(date) AS quarter,

    dayofweek(date) IN (1, 7) AS is_weekend

FROM (
    SELECT DISTINCT
        CAST(date AS DATE) AS date
    FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly
)
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dim_date"))